In [1]:
# Importar las librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
# URL del archivo en el bucket público de AWS
url = "https://hybridge-education-machine-learning-datasets.s3.us-east-1.amazonaws.com/Fraud.csv"
df = pd.read_csv(url)

In [3]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [4]:
len(df)

6362620

# ¡¡¡6,362,620 OBSERVACIONES!!!

In [5]:
# Preprocesamiento y selección de características

# Eliminar columnas de identificación que no sirven para el modelo matemático
cols_to_drop = [c for c in ['nameOrig', 'nameDest', 'isFlaggedFraud'] if c in df.columns]
df_cleaned = df.drop(columns=cols_to_drop)

# Codificar la variable categórica 'type' (tipo de transacción) en columnas binarias
if 'type' in df_cleaned.columns:
    df_encoded = pd.get_dummies(df_cleaned, columns=['type'], drop_first=True)
else:
    df_encoded = df_cleaned.copy()

# Separar las características (X) y la variable objetivo (y)
target_col = 'isFraud' if 'isFraud' in df_encoded.columns else df_encoded.columns[-1]
X = df_encoded.drop(columns=[target_col])
y = df_encoded[target_col]

print(f"Estructura final de X: {X.shape}")

Estructura final de X: (6362620, 10)


In [6]:
# División en entrenamiento y prueba y escalado

# Dividir los datos (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Inicializar y aplicar el escalado estándar
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Muestras de entrenamiento: {X_train_scaled.shape[0]}")
print(f"Muestras de prueba: {X_test_scaled.shape[0]}")

Muestras de entrenamiento: 5090096
Muestras de prueba: 1272524


In [7]:
# Entrenamiento del modelo de regresión logística

# Inicializar el clasificador
model = LogisticRegression(max_iter=1000, random_state=42)

# Entrenar con los datos ya escalados
model.fit(X_train_scaled, y_train)

print("Modelo entrenado correctamente.")

Modelo entrenado correctamente.


In [8]:
# Predicción y evaluación final

# Realizar las predicciones sobre el conjunto de prueba
y_pred = model.predict(X_test_scaled)

# Calcular matriz de confusión y reporte de métricas
conf_matrix = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred)

print("EVALUACIÓN FINAL DEL MODELO")
print("Matriz de Confusión:")
print(conf_matrix)
print("\nReporte de Clasificación Completo:")
print(report)

EVALUACIÓN FINAL DEL MODELO
Matriz de Confusión:
[[1270817      64]
 [    994     649]]

Reporte de Clasificación Completo:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.91      0.40      0.55      1643

    accuracy                           1.00   1272524
   macro avg       0.95      0.70      0.78   1272524
weighted avg       1.00      1.00      1.00   1272524

